## Benchmark Meta data

In [1]:
# read all .csvs from ../../../applicayions/*.csv

# this looks like 
# benchmark,n_qubits,depth,width,n_clbits,n_ops,n_gates,n_parameters,n_connected_components,n_measure_ops,single_qubit_gates,two_qubit_gates,three_qubit_gates,four_qubit_gates,gate_barrier,gate_ccx,gate_cp,gate_cu,gate_cx,gate_cz,gate_h,gate_measure,gate_p,gate_rx,gate_ry,gate_rz,gate_s,gate_swap,gate_u2
# ghz,2,3,2,2,4,4,0,,2,3,1,0,0,0,0,0,0,1,0,1,2,0,0,0,0,0,0,0

import glob
import pandas as pd
from pprint import pprint

csv_files = glob.glob("../../../applications/*.csv")
pprint(csv_files)
if not csv_files:
    print("No CSV files found in ../../../applications/")
else:
    benchmark_meta_df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    if benchmark_meta_df.empty:
        print("[warn] No data loaded from benchmark CSVs.")
    else:
        print(f"Loaded benchmark_meta_df shape: {benchmark_meta_df.shape}")

# show all column names
pprint(benchmark_meta_df.columns.tolist())

# show top 10 rows
display(benchmark_meta_df.head(5))

['../../../applications/meta15to18.csv',
 '../../../applications/meta6to10.csv',
 '../../../applications/meta2to5.csv',
 '../../../applications/meta11to14.csv',
 '../../../applications/meta19to20.csv']
Loaded benchmark_meta_df shape: (84, 59)
['benchmark',
 'n_qubits',
 'depth',
 'width',
 'n_clbits',
 'n_ops',
 'n_gates',
 'n_parameters',
 'n_connected_components',
 'n_measure_ops',
 'single_qubit_gates',
 'two_qubit_gates',
 'three_qubit_gates',
 'four_qubit_gates',
 'num_separable_circuits',
 'num_tensor_factors',
 'num_serial_layers',
 'qc_width_total',
 'num_registers',
 'num_cregs',
 'num_ancillas',
 'idle_wires',
 'global_phase',
 'dag_size_ops',
 'dag_edge_count',
 'longest_path_len',
 'num_control_flow_ops',
 'num_barriers',
 'num_distinct_twoq_pairs',
 'touch_min',
 'touch_max',
 'touch_avg',
 'clifford_count',
 'nonclifford_count',
 'measure_fraction',
 'total_gate_params',
 'avg_gate_params',
 'std_gate_params',
 'min_gate_params',
 'max_gate_params',
 'num_controlled_gates

,benchmark,n_qubits,depth,width,n_clbits,n_ops,n_gates,n_parameters,n_connected_components,n_measure_ops,...,gate_cz,gate_h,gate_measure,gate_p,gate_rx,gate_ry,gate_rz,gate_s,gate_swap,gate_u2
0,ghz,15,16,15,15,30,30,0,1,15,...,0,1,15,0,0,0,0,0,0,0
1,ghz,16,17,16,16,32,32,0,1,16,...,0,1,16,0,0,0,0,0,0,0
2,ghz,17,18,17,17,34,34,0,1,17,...,0,1,17,0,0,0,0,0,0,0
3,ghz,18,19,18,18,36,36,0,1,18,...,0,1,18,0,0,0,0,0,0,0
4,ham,15,46,15,15,102,102,0,1,15,...,0,30,15,0,0,0,29,0,0,0


## Runtime data for different backends via QFw

In [2]:
# Imports and data loading
import os, json
import pandas as pd
from pprint import pprint

json_path = os.path.abspath(os.path.join("../..", "qfw_unified_data.json"))
csv_path  = os.path.abspath(os.path.join("../..", "qfw_unified_summary.csv"))

print("JSON:", json_path)
print("CSV: ", csv_path)

with open(json_path, "r") as f:
    data = json.load(f)

backend_runtime_df = pd.read_csv(csv_path)

print("\nJSON top-level keys:", list(data.keys())[:10])
print("CSV shape:", backend_runtime_df.shape)

# show all column names
pprint(backend_runtime_df.columns.tolist())

# Show top 5 rows of the CSV data
backend_runtime_df.head(5)

JSON: /lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/results_analysis/qfw_unified_data.json
CSV:  /lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/results_analysis/qfw_unified_summary.csv

JSON top-level keys: ['ghz', 'ham', 'tfim', 'hhl', 'qaoa', 'dqaoa']
CSV shape: (307, 11)
['benchmark',
 'size',
 'backend',
 'sub_backend',
 'device',
 'run_mode',
 'n_nodes',
 'n_processes',
 'samples',
 'mean_ms',
 'std_ms']


,benchmark,size,backend,sub_backend,device,run_mode,n_nodes,n_processes,samples,mean_ms,std_ms
0,ghz,16,qiskitaer,statevector,CPU,sync,2,2,3,7244.896650,280.149080
1,ghz,16,qiskitaer,statevector,CPU,sync,1,4,3,6820.161263,473.606571
2,ghz,16,qiskitaer,matrix,CPU,sync,1,4,3,6648.384094,427.302495
3,ghz,16,qiskitaer,matrix,CPU,sync,2,2,3,6563.753605,416.044224
4,ghz,16,qiskitaer,automatic,CPU,sync,1,4,3,6828.919172,360.575876


Given all benchmark meta data, `n_processes`, and `n_nodes`, we want to label the best backend (lowest runtime)!

**Goal:**  
Append new columns into a copy of `benchmark_meta_df` (call it `bench_data_df`):

- `n_processes`
- `n_nodes`
- `best_backend`
- `best_runtime` (for this best_backend)

**Motivation:**  
Once we train a model, given benchmark meta and `n_processes`/`n_nodes`, we can predict two things:

1. **Best Backend**  
	_Classification problem_: Predict the backend with the lowest runtime.

2. **Best Runtime for a Given Backend Choice**  
	_Regression problem_: Predict the expected runtime for a given backend.

In [3]:
# Prepare two DataFrames: one for classification (best backend), one for regression (runtime estimation)
# --- Classification: best_backend_df ---
# For each (benchmark, size, n_nodes, n_processes), find the backend with the lowest mean_ms (runtime)
group_cols = ['benchmark', 'size', 'n_nodes', 'n_processes']
idx = backend_runtime_df.groupby(group_cols)['mean_ms'].idxmin()
best_backend_df = backend_runtime_df.loc[idx, group_cols + ['backend', 'mean_ms']]
best_backend_df = best_backend_df.rename(columns={'backend': 'best_backend', 'mean_ms': 'best_runtime'})

# Merge with benchmark_meta_df on 'benchmark' and 'size' (size <-> n_qubits)
best_backend_df = pd.merge(
    best_backend_df,
    benchmark_meta_df.assign(size=benchmark_meta_df['n_qubits'].astype('Int64')),
    how='left',
    on=['benchmark', 'size']
 )

print("[Classification] best_backend_df columns:")
pprint(best_backend_df.columns.tolist())
print(f"[Classification] Rows: {len(best_backend_df)}")
display(best_backend_df.head(5))

# --- Regression: estimate_runtime_df ---
# Each row is a (benchmark, size, n_nodes, n_processes, backend) tuple, with mean_ms as the label
estimate_runtime_df = backend_runtime_df.merge(
    benchmark_meta_df.assign(size=benchmark_meta_df['n_qubits'].astype('Int64')),
    how='left',
    on=['benchmark', 'size']
)

print("[Regression] estimate_runtime_df columns:")
pprint(estimate_runtime_df.columns.tolist())
print(f"[Regression] Rows: {len(estimate_runtime_df)}")
display(estimate_runtime_df.head(5))

[Classification] best_backend_df columns:
['benchmark',
 'size',
 'n_nodes',
 'n_processes',
 'best_backend',
 'best_runtime',
 'n_qubits',
 'depth',
 'width',
 'n_clbits',
 'n_ops',
 'n_gates',
 'n_parameters',
 'n_connected_components',
 'n_measure_ops',
 'single_qubit_gates',
 'two_qubit_gates',
 'three_qubit_gates',
 'four_qubit_gates',
 'num_separable_circuits',
 'num_tensor_factors',
 'num_serial_layers',
 'qc_width_total',
 'num_registers',
 'num_cregs',
 'num_ancillas',
 'idle_wires',
 'global_phase',
 'dag_size_ops',
 'dag_edge_count',
 'longest_path_len',
 'num_control_flow_ops',
 'num_barriers',
 'num_distinct_twoq_pairs',
 'touch_min',
 'touch_max',
 'touch_avg',
 'clifford_count',
 'nonclifford_count',
 'measure_fraction',
 'total_gate_params',
 'avg_gate_params',
 'std_gate_params',
 'min_gate_params',
 'max_gate_params',
 'num_controlled_gates',
 'num_unitary_factors',
 'dag_duration',
 'qc_duration',
 'gate_barrier',
 'gate_ccx',
 'gate_cp',
 'gate_cu',
 'gate_cx',
 'ga

,benchmark,size,n_nodes,n_processes,best_backend,best_runtime,n_qubits,depth,width,n_clbits,...,gate_cz,gate_h,gate_measure,gate_p,gate_rx,gate_ry,gate_rz,gate_s,gate_swap,gate_u2
0,dqaoa,30,1,56,nwqsim,136000.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,dqaoa,40,1,56,nwqsim,201000.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ghz,4,1,4,nwqsim,437.901656,4.0,5.0,4.0,4.0,...,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ghz,8,1,4,nwqsim,454.950809,8.0,9.0,8.0,8.0,...,0.0,1.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ghz,12,1,4,nwqsim,459.396839,12.0,13.0,12.0,12.0,...,0.0,1.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


[Regression] estimate_runtime_df columns:
['benchmark',
 'size',
 'backend',
 'sub_backend',
 'device',
 'run_mode',
 'n_nodes',
 'n_processes',
 'samples',
 'mean_ms',
 'std_ms',
 'n_qubits',
 'depth',
 'width',
 'n_clbits',
 'n_ops',
 'n_gates',
 'n_parameters',
 'n_connected_components',
 'n_measure_ops',
 'single_qubit_gates',
 'two_qubit_gates',
 'three_qubit_gates',
 'four_qubit_gates',
 'num_separable_circuits',
 'num_tensor_factors',
 'num_serial_layers',
 'qc_width_total',
 'num_registers',
 'num_cregs',
 'num_ancillas',
 'idle_wires',
 'global_phase',
 'dag_size_ops',
 'dag_edge_count',
 'longest_path_len',
 'num_control_flow_ops',
 'num_barriers',
 'num_distinct_twoq_pairs',
 'touch_min',
 'touch_max',
 'touch_avg',
 'clifford_count',
 'nonclifford_count',
 'measure_fraction',
 'total_gate_params',
 'avg_gate_params',
 'std_gate_params',
 'min_gate_params',
 'max_gate_params',
 'num_controlled_gates',
 'num_unitary_factors',
 'dag_duration',
 'qc_duration',
 'gate_barrier',


,benchmark,size,backend,sub_backend,device,run_mode,n_nodes,n_processes,samples,mean_ms,...,gate_cz,gate_h,gate_measure,gate_p,gate_rx,gate_ry,gate_rz,gate_s,gate_swap,gate_u2
0,ghz,16,qiskitaer,statevector,CPU,sync,2,2,3,7244.896650,...,0.0,1.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ghz,16,qiskitaer,statevector,CPU,sync,1,4,3,6820.161263,...,0.0,1.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ghz,16,qiskitaer,matrix,CPU,sync,1,4,3,6648.384094,...,0.0,1.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ghz,16,qiskitaer,matrix,CPU,sync,2,2,3,6563.753605,...,0.0,1.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ghz,16,qiskitaer,automatic,CPU,sync,1,4,3,6828.919172,...,0.0,1.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# save them as csvs under final_data/ "create directory if it doesn't exist in current working directory"
os.makedirs("final_data", exist_ok=True)
best_backend_df.to_csv("final_data/best_backend_df.csv", index=False)
estimate_runtime_df.to_csv("final_data/estimate_runtime_df.csv", index=False)